# DIMER Notebook: Open-Vocabulary Detection and Counting

**Profile:** `MULTI-CAPABILITY`  
**Mode:** `WORKSHOP`  
**Notebook specification:** `2.1`  
**Status:** Candidate comparative carrier  
**Default tier:** `STANDARD`  
**Canonical runtime:** NVIDIA Tesla T4 or equivalent

This notebook links two prompt-driven vision tasks without pretending they are identical.

## Capability A — open-vocabulary detection

Compare:

- **Grounding DINO Tiny**
- **OWLv2 Base P16 Ensemble**

on the same images, text phrases, boxes, score floor, and notebook-owned detector evaluator.

## Capability B — open-world counting

Run **CountGD** in three prompt modes:

- text only;
- visual exemplars only;
- text + exemplars.

Evaluate not only count error, but also whether the model localized the correct target objects.

## Capability C — detection as counting

Take the Grounding DINO / OWLv2 target detections, apply one shared class-wise NMS rule, and simply count retained boxes.

This asks:

> When is a dedicated counting model actually different from “detect the objects and count the boxes”?

### Core experiment

The canonical sample is one shared set of **44 deterministic target+distractor scenes**:

- 24 train
- 8 validation
- 12 independent test

Every scene includes exact:

- target boxes and centers;
- distractor boxes and centers;
- target count;
- target text label;
- distractor text label;
- three target exemplar boxes.

### Execution tiers

`STANDARD` runs the frozen three-model comparison plus detector-as-counter.

`FULL` additionally runs the release-grade bounded adaptation paths for:

- Grounding DINO
- CountGD

OWLv2 remains frozen because its live DIMER E2E adaptation carrier is still candidate.

### Candidate-stage source boundary

Grounding DINO and OWLv2 frozen inference use only pinned Transformers-format upstream snapshots.

CountGD's current qualified model implementation is substantial and is not available as a normal pip package. This candidate notebook therefore stages the exact `countgd_pipeline` Python files from the immutable DIMER carrier commit, verifies each Git blob identity, and executes them locally.

`FULL` similarly stages the Grounding DINO carrier for its bounded adapter implementation.

**Release hardening gate:** carry those pinned carrier modules directly inside notebook cells before promotion to release-grade standalone 2.1. The model/data/evaluation contracts below do not need to change.

All built-in measurements are **tutorial/sample-sanity evidence**, not deployment benchmarks.

## How to use this notebook

**Who it is for.** Learners who can run Python cells in Colab/Jupyter and have seen basic object detection; this notebook extends that idea to open-world counting.

**Runtime.** Use the documented GPU runtime for the comparative detection/counting path.

**How to run it.**
1. Select the documented runtime/accelerator.
2. Choose **Run all** for the canonical path; leave the default settings unchanged on your first pass.
3. Read the explanatory markdown while the notebook runs.
4. Sections marked **Infrastructure** support reproducibility, model acquisition, or orchestration. Run those cells as written; understanding their implementation is not a learning objective.

### Task at a glance

`image + concept prompt → detection and/or counting model → locations/count → detection and counting evaluation`

### Roadmap

1. Understand the task and its input/output contract.
2. Inspect and validate the built-in data or inputs.
3. Establish the baseline/reference behavior.
4. Run the model or multi-model comparison.
5. Inspect errors, disagreements, robustness, and/or resource tradeoffs.
6. Try one controlled change and explain what changed.
7. Write an evidence-based conclusion; optionally continue with BYOD.

### What successful execution looks like

You should finish with a validated input/sample, the notebook's principal baseline/reference, model outputs and evaluation results, at least one diagnostic or qualitative comparison, and machine-readable results/provenance where supported. Exact values can vary slightly across supported runtimes; focus on the defined metrics and the observed pattern.


## 0. Learning goals

Participants should be able to:

1. explain what “open vocabulary” does and does not mean;
2. compare Grounding DINO and OWLv2 under one text-query contract;
3. understand prompt wording as part of the model input distribution;
4. identify duplicate detections and negative-prompt false positives;
5. read AP/AP50/AP75 and phrase-level recall;
6. distinguish open-vocabulary detection from open-world counting;
7. read MAE, RMSE and normalized absolute error;
8. detect compensating counting errors using box/point localization;
9. compare text, exemplar, and combined CountGD prompting;
10. convert detection outputs into counts under a controlled NMS/threshold rule;
11. preserve validation-owned threshold selection and freeze-before-test;
12. understand why adaptation can improve one domain while regressing another.

## 1. Notebook controls

In [ ]:
# @title Workshop controls
WORKSHOP_TIER = "STANDARD"  # @param ["STANDARD", "FULL"]

USE_BYOD = False            # @param {type:"boolean"}
BYOD_ZIP_PATH = ""          # @param {type:"string"}

OUTPUT_DIR = "outputs"      # @param {type:"string"}

DETECTION_SCORE_FLOOR = 0.05
DETECTION_CANDIDATE_CAP = 900
COMMON_NMS_IOU = 0.50

COUNTGD_THRESHOLD = 0.23
COUNT_THRESHOLD_GRID = (0.05, 0.10, 0.15, 0.20, 0.30, 0.40)
COUNTGD_THRESHOLD_GRID = (0.15, 0.20, 0.23, 0.30, 0.40)

ABSENT_PROMPT = "purple star"

GROUNDING_EPOCHS = 6
GROUNDING_LR = 5e-5
GROUNDING_BATCH_SIZE = 4

COUNTGD_EPOCHS = 4
COUNTGD_LR = 2e-4

if WORKSHOP_TIER not in {"STANDARD", "FULL"}:
    raise ValueError("WORKSHOP_TIER must be STANDARD or FULL")

from pathlib import Path
OUTPUT_ROOT = Path(OUTPUT_DIR)
WORK_ROOT = Path("work")

for p in [
    OUTPUT_ROOT / "data",
    OUTPUT_ROOT / "detection" / "validation",
    OUTPUT_ROOT / "detection" / "test",
    OUTPUT_ROOT / "counting" / "validation",
    OUTPUT_ROOT / "counting" / "test",
    OUTPUT_ROOT / "cross_capability",
    OUTPUT_ROOT / "adaptation" / "grounding_dino",
    OUTPUT_ROOT / "adaptation" / "countgd",
    OUTPUT_ROOT / "frozen",
    OUTPUT_ROOT / "artifacts",
    OUTPUT_ROOT / "figures",
    OUTPUT_ROOT / "new_data",
    OUTPUT_ROOT / "provenance",
    WORK_ROOT / "models" / "grounding_dino",
    WORK_ROOT / "models" / "owlv2",
    WORK_ROOT / "models" / "countgd",
    WORK_ROOT / "models" / "bert",
    WORK_ROOT / "carrier_src",
]:
    p.mkdir(parents=True, exist_ok=True)

print({
    "tier": WORKSHOP_TIER,
    "detection_score_floor": DETECTION_SCORE_FLOOR,
    "common_nms_iou": COMMON_NMS_IOU,
    "countgd_threshold": COUNTGD_THRESHOLD,
})

## 2. Install the pinned shared runtime

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
# @title Install runtime
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    "torch==2.14.0",
    "torchvision==0.29.0",
    "torchaudio==2.11.0",
    "transformers==4.57.6",
    "huggingface-hub==0.36.2",
    "safetensors==0.8.0",
    "numpy==2.5.3",
    "pillow==11.3.0",
    "scipy==1.18.1",
    "matplotlib>=3.9,<3.11",
    "pandas>=2.2,<3.1",
]

SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"

# Hosted kernels such as Colab import NumPy at startup. Replacing a loaded module on disk would force a manual
# restart (Notebook Spec RUN10), so a NumPy 2.x that is already loaded is kept and recorded instead of reinstalled.
NUMPY_PRELOADED = None
if "numpy" in sys.modules and str(getattr(sys.modules["numpy"], "__version__", "")).startswith("2."):
    NUMPY_PRELOADED = sys.modules["numpy"].__version__
    PINS = [f"numpy=={NUMPY_PRELOADED}" if pin.startswith("numpy==") else pin for pin in PINS]

if not SKIP_INSTALL:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *PINS],
        check=True,
    )
    importlib.invalidate_caches()

# Fail closed rather than run with a module whose files were replaced underneath it.
stale = [
    (name, sys.modules[name].__version__, importlib.metadata.version(name))
    for name in ("numpy", "torch")
    if name in sys.modules
    and str(sys.modules[name].__version__).split("+")[0] != importlib.metadata.version(name).split("+")[0]
]
if stale:
    raise RuntimeError(
        "The kernel had already imported packages that the pinned install replaced on disk. "
        "Restart the Python session and choose Run all again (Colab: Runtime > Restart session; "
        f"do not delete the runtime, which discards the installed pins). Stale modules: {stale}"
    )
print("numpy:", "preloaded by the host kernel" if NUMPY_PRELOADED else "pinned install", sys.modules.get("numpy") and sys.modules["numpy"].__version__)

import numpy as np
import pandas as pd
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageOps

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "transformers": importlib.metadata.version("transformers"),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})

# 3. Immutable model identities

## Grounding DINO Tiny

`IDEA-Research/grounding-dino-tiny`

Revision:

`a2bb814dd30d776dcf7e30523b00659f4f141c71`

SafeTensors:

- 689,359,096 bytes
- SHA-256 `1a2412ef99bd74bcd3c2a246fa1e48581f8889a1300c9051974741314fc042f3`

Architecture:

- Swin-T image backbone
- BERT text encoder
- multimodal enhancer
- DETR-style decoder
- 900 object queries

## OWLv2 Base P16 Ensemble

`google/owlv2-base-patch16-ensemble`

Revision:

`cfd3195ba4ea9592eec887ded089f4c08eff231d`

SafeTensors:

- 619,918,824 bytes
- SHA-256 `e1e130b9e404cf91a75ad45644c1da9d7fa5284085eecc864266a6923efb99e7`

Architecture:

- CLIP ViT-B/16 image tower
- CLIP text tower
- 960×960 input
- 60×60 = 3,600 patch candidates

## CountGD

Pinned model:

`nikigoli/countgd`

Space revision:

`6e82e59569a84ee5c6aafa35d396f2d2bee57be2`

DIMER converted model:

`countgd.safetensors`

- 937,560,480 bytes
- SHA-256 `8e44867b951e3a4205d918e022b78bc5fea218fd17c1851b864a01c421d2d443`

Parameters:

approximately 233.36M.

CountGD extends Grounding DINO with visual exemplar tokens pooled from exemplar boxes.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
# @title Exact snapshot manifests
GROUNDING_MANIFEST = {
    "format":"dimer_hf_snapshot","formatVersion":1,
    "modelKey":"grounding-dino-tiny",
    "modelId":"IDEA-Research/grounding-dino-tiny",
    "revision":"a2bb814dd30d776dcf7e30523b00659f4f141c71",
    "files":[
        ["README.md",2580,"cf46f74c7b6850f1d5cbe406028324d8798148726016d46a69a365b4a2d3e89f"],
        ["added_tokens.json",82,"909e96cb32d92ce728a01bc99850cbba26196d74115c17ebeb019275412588f2"],
        ["config.json",1644,"eec82c5ab66e16df12a9a212e68ac011779927c2536cf9078658e35d85f0c67a"],
        ["model.safetensors",689359096,"1a2412ef99bd74bcd3c2a246fa1e48581f8889a1300c9051974741314fc042f3"],
        ["preprocessor_config.json",457,"8454179ba95e2ad22947835aad7b45862a601fc0055ab88bf1ee70892d3aea60"],
        ["special_tokens_map.json",125,"b6d346be366a7d1d48332dbc9fdf3bf8960b5d879522b7799ddba59e76237ee3"],
        ["tokenizer.json",711396,"d241a60d5e8f04cc1b2b3e9ef7a4921b27bf526d9f6050ab90f9267a1f9e5c66"],
        ["tokenizer_config.json",1237,"d40ab645b68211910b9170d22433d43186a6ec8ee6fd10ba170524b25bf4fb56"],
        ["vocab.txt",231508,"07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"],
    ],
}
OWLV2_MANIFEST = {
    "format":"dimer_hf_snapshot","formatVersion":1,
    "modelKey":"owlv2-base-patch16-ensemble",
    "modelId":"google/owlv2-base-patch16-ensemble",
    "revision":"cfd3195ba4ea9592eec887ded089f4c08eff231d",
    "files":[
        ["README.md",4838,"7c7426bc5ec939a42d1f96fb093031b6263400cceac4129ebb941a0c8c11b9b9"],
        ["added_tokens.json",67,"e5dc0da35d20111e8ff3fdfc03682beca23d5f94ed74331bce81786b2636a24f"],
        ["config.json",414,"ba9df8c25a4b8461887dd0a93d9252c9cd84697fe8d49a9d8794ce409af9acb2"],
        ["merges.txt",524619,"9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a"],
        ["model.safetensors",619918824,"e1e130b9e404cf91a75ad45644c1da9d7fa5284085eecc864266a6923efb99e7"],
        ["preprocessor_config.json",425,"cf3e396635b797ee1a464e1b2836e98748f8edac19e89aaa2c93b55ac15b0064"],
        ["special_tokens_map.json",121,"d6e2b9cf664efbad2d22998b8d3da986abcbeed3e0825ad33605c9401f9cf73e"],
        ["tokenizer_config.json",1100,"b55cda6198e152ded427c8a9b3faf1cccf27a7fa080697a62f6ff143f511f44f"],
        ["vocab.json",1059962,"e089ad92ba36837a0d31433e555c8f45fe601ab5c221d4f607ded32d9f7a4349"],
    ],
}
COUNTGD_BASE_MANIFEST = {
    "format":"dimer_hf_snapshot","formatVersion":1,
    "modelKey":"countgd","modelId":"nikigoli/countgd","repoType":"space",
    "revision":"6e82e59569a84ee5c6aafa35d396f2d2bee57be2",
    "files":[
        {"path":"README.md","bytes":526,"sha256":"f67b0f10bb3d47ec0b810624f2e1c71ffcc026458936e90e5e203fe08408d150"},
        {"path":"checkpoint_best_regular.pth","bytes":1250122522,"sha256":"c1bab864b17db345b4c6e3aaabb5765bc2c0a90d0bc8defb5e664a74a50aa126"},
    ],
    "totalBytes":1250123048,
}
COUNTGD_BERT_MANIFEST = {
    "format":"dimer_hf_snapshot","formatVersion":1,
    "modelKey":"bert-base-uncased","modelId":"google-bert/bert-base-uncased",
    "revision":"86b5e0934494bd15c9632b12f734a8a67f723594",
    "files":[
        {"path":"LICENSE","bytes":11356,"sha256":"43070e2d4e532684de521b885f385d0841030efa2b1a20bafb76133a5e1379c1"},
        {"path":"README.md","bytes":10517,"sha256":"9187b6018ea0010d884e78e098e328faa1b88b301570d0cce606bb35e4067e17"},
        {"path":"config.json","bytes":570,"sha256":"7160e1553ad2ca51d8c1cb066be533db31826e12d173824c1bb0cb1a4f187d20"},
        {"path":"tokenizer.json","bytes":466062,"sha256":"ce64fce797c24f68df90b40a3f74f579b336a493db14bd583fd520ea0d8c9a98"},
        {"path":"tokenizer_config.json","bytes":48,"sha256":"a025160ef0431f1a392f6f050c1310f4c5d9fb6f275932dbccba73c4d214bf10"},
        {"path":"vocab.txt","bytes":231508,"sha256":"07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"},
    ],
    "totalBytes":720061,
}

# 4. Shared 44-scene target+distractor generator

This is derived from the live CountGD synthetic fixture, with one important extension:

> the generator now retains the distractor boxes and centers that it already knows while drawing them.

Rendering semantics remain the same.

Scene structure:

- 512×384 RGB
- 6–40 target objects
- 4–20 distractors
- 300 clutter specks
- target/distractor chosen from colors × shapes
- three exemplar boxes sampled from target instances

The same source image is given to all three models.

In [ ]:
# @title Shared synthetic generator
import random
import math
import hashlib
import io
import json

SYNTHETIC_SIZE=(512,384)
SHAPES=("circle","square","triangle")
COLOURS={
    "red":(200,40,40),
    "blue":(40,70,200),
    "green":(40,150,60),
    "yellow":(220,190,30),
}
TARGET_RANGE=(6,40)
DISTRACTOR_RANGE=(4,20)
RADIUS_RANGE=(7,14)
SPECKS=300
SEED_BASE={"train":1000,"validation":2000,"test":3000}
SPLIT_SIZES={"train":24,"validation":8,"test":12}

def _draw_shape(draw,shape,box,fill):
    x0,y0,x1,y1=box
    if shape=="circle":
        draw.ellipse(box,fill=fill)
    elif shape=="square":
        draw.rectangle(box,fill=fill)
    else:
        draw.polygon([((x0+x1)//2,y0),(x1,y1),(x0,y1)],fill=fill)

def synthetic_scene(seed,n_targets=None):
    if isinstance(seed,bool) or not isinstance(seed,int) or seed<0:
        raise ValueError("seed must be a non-negative int")
    rng=random.Random(seed)
    width,height=SYNTHETIC_SIZE
    image=Image.new("RGB",SYNTHETIC_SIZE,tuple(rng.randint(150,225) for _ in range(3)))
    draw=ImageDraw.Draw(image)
    for _ in range(SPECKS):
        draw.point(
            (rng.randrange(width),rng.randrange(height)),
            fill=tuple(rng.randint(90,250) for _ in range(3)),
        )

    target_shape=rng.choice(SHAPES)
    target_colour=rng.choice(sorted(COLOURS))
    distractor_shape=rng.choice([s for s in SHAPES if s!=target_shape])
    distractor_colour=rng.choice([c for c in sorted(COLOURS) if c!=target_colour])

    wanted_targets=rng.randint(*TARGET_RANGE) if n_targets is None else int(n_targets)
    wanted_distractors=rng.randint(*DISTRACTOR_RANGE)
    radius=rng.randint(*RADIUS_RANGE)

    placed=[]
    target_boxes=[]
    distractor_boxes=[]

    for kind,wanted in (("target",wanted_targets),("distractor",wanted_distractors)):
        tries=0
        while wanted>0 and tries<5000:
            tries+=1
            size=radius+rng.randint(-2,2)
            cx=rng.randint(size+2,width-size-3)
            cy=rng.randint(size+2,height-size-3)
            if any((cx-px)**2+(cy-py)**2 <= (size+ps+3)**2 for px,py,ps in placed):
                continue
            placed.append((cx,cy,size))
            box=[cx-size,cy-size,cx+size,cy+size]
            if kind=="target":
                _draw_shape(draw,target_shape,box,COLOURS[target_colour])
                target_boxes.append([float(box[0]),float(box[1]),float(box[2]+1),float(box[3]+1)])
            else:
                _draw_shape(draw,distractor_shape,box,COLOURS[distractor_colour])
                distractor_boxes.append([float(box[0]),float(box[1]),float(box[2]+1),float(box[3]+1)])
            wanted-=1

    exemplars=[list(b) for b in rng.sample(target_boxes,k=min(3,len(target_boxes)))]

    buf=io.BytesIO()
    image.save(buf,format="PNG")
    pixel_sha=hashlib.sha256(buf.getvalue()).hexdigest()

    record={
        "id":f"synth-{seed:05d}",
        "image":image,
        "target_label":f"{target_colour} {target_shape}",
        "target_boxes":target_boxes,
        "target_points":[[(b[0]+b[2])/2,(b[1]+b[3])/2] for b in target_boxes],
        "target_count":len(target_boxes),
        "exemplars":exemplars,
        "distractor_label":f"{distractor_colour} {distractor_shape}",
        "distractor_boxes":distractor_boxes,
        "distractor_points":[[(b[0]+b[2])/2,(b[1]+b[3])/2] for b in distractor_boxes],
        "distractor_count":len(distractor_boxes),
        "pixel_sha256":pixel_sha,
    }
    return record

def build_dataset():
    out={}
    for split,n in SPLIT_SIZES.items():
        out[split]=[]
        for i in range(n):
            record=synthetic_scene(SEED_BASE[split]+i)
            record["split"]=split
            out[split].append(record)
    return out

splits=build_dataset()
train_records=splits["train"]
validation_records=splits["validation"]
test_records=splits["test"]

seen={}
for split,records in splits.items():
    for r in records:
        if r["pixel_sha256"] in seen:
            raise RuntimeError(f"pixel leak across {seen[r['pixel_sha256']]} and {split}")
        seen[r["pixel_sha256"]]=split

print({
    split:{
        "scenes":len(records),
        "targets":sum(r["target_count"] for r in records),
        "distractors":sum(r["distractor_count"] for r in records),
    }
    for split,records in splits.items()
})

In [ ]:
# @title Export sample manifest
def record_manifest(r):
    return {
        "id":r["id"],
        "split":r["split"],
        "pixel_sha256":r["pixel_sha256"],
        "target_label":r["target_label"],
        "target_boxes":r["target_boxes"],
        "target_count":r["target_count"],
        "exemplars":r["exemplars"],
        "distractor_label":r["distractor_label"],
        "distractor_boxes":r["distractor_boxes"],
        "distractor_count":r["distractor_count"],
    }

dataset_manifest={
    "generator":"dimer-open-vocab-counting-v1",
    "size":list(SYNTHETIC_SIZE),
    "seed_bases":SEED_BASE,
    "split_sizes":SPLIT_SIZES,
    "records":[record_manifest(r) for part in splits.values() for r in part],
}
dataset_manifest["dataset_sha256"]=hashlib.sha256(
    json.dumps(dataset_manifest["records"],sort_keys=True,separators=(",",":")).encode()
).hexdigest()

(OUTPUT_ROOT/"data"/"dataset_manifest.json").write_text(
    json.dumps(dataset_manifest,indent=2),encoding="utf-8"
)

print("dataset_sha256:",dataset_manifest["dataset_sha256"])

In [ ]:
# @title Shared scene gallery
fig,axes=plt.subplots(2,3,figsize=(13,8))
for ax,r in zip(axes.ravel(),validation_records[:6]):
    ax.imshow(r["image"])
    for box in r["target_boxes"]:
        x0,y0,x1,y1=box
        ax.add_patch(plt.Rectangle((x0,y0),x1-x0,y1-y0,fill=False,linewidth=1.5))
    ax.set_title(f"{r['target_label']} × {r['target_count']}\nvs {r['distractor_label']} × {r['distractor_count']}")
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT/"figures"/"shared_scene_gallery.png",dpi=150,bbox_inches="tight")
plt.show()

# 5. Common detection evaluator

Every detection model is normalized to:

```text
{"box":[x0,y0,x1,y1], "label":<one requested phrase>, "score":float}
```

The notebook then owns:

- AP@[.50:.95]
- AP50
- AP75
- recall@50
- false positives/image
- missed objects/image
- mean matched IoU
- duplicate rate

This avoids comparing model-specific evaluation implementations.

In [ ]:
# @title Detection metrics and common NMS
from torchvision.ops import nms

IOU_THRESHOLDS=tuple(round(0.50+0.05*i,2) for i in range(10))

def box_iou(a,b):
    ax0,ay0,ax1,ay1=map(float,a); bx0,by0,bx1,by1=map(float,b)
    ix0,iy0=max(ax0,bx0),max(ay0,by0)
    ix1,iy1=min(ax1,bx1),min(ay1,by1)
    inter=max(0,ix1-ix0)*max(0,iy1-iy0)
    area_a=max(0,ax1-ax0)*max(0,ay1-ay0)
    area_b=max(0,bx1-bx0)*max(0,by1-by0)
    union=area_a+area_b-inter
    return inter/union if union>0 else 0.0

def detection_reference(record):
    return {
        "boxes":record["target_boxes"]+record["distractor_boxes"],
        "labels":[record["target_label"]]*record["target_count"]+
                 [record["distractor_label"]]*record["distractor_count"],
    }

def common_ap(predictions,records,iou_thresholds=IOU_THRESHOLDS):
    refs=[detection_reference(r) for r in records]
    vocabulary=sorted({label for ref in refs for label in ref["labels"]})
    recall_points=np.linspace(0,1,101)
    per_threshold={}

    for threshold in iou_thresholds:
        per_class={}
        for cls in vocabulary:
            scored=[]
            n_refs=0
            for dets,ref in zip(predictions,refs):
                ref_boxes=[b for b,l in zip(ref["boxes"],ref["labels"]) if l==cls]
                n_refs+=len(ref_boxes)
                claimed=[False]*len(ref_boxes)
                candidates=sorted(
                    (d for d in dets if d["label"]==cls),
                    key=lambda d:-float(d["score"])
                )
                for det in candidates:
                    best,best_iou=-1,0.0
                    for j,b in enumerate(ref_boxes):
                        if claimed[j]:continue
                        value=box_iou(det["box"],b)
                        if value>best_iou:best,best_iou=j,value
                    hit=best>=0 and best_iou>=threshold
                    if hit:claimed[best]=True
                    scored.append((float(det["score"]),hit))
            if n_refs==0:continue
            if not scored:
                per_class[cls]=0.0
                continue
            scored.sort(key=lambda x:-x[0])
            tp=np.cumsum([1 if h else 0 for _,h in scored])
            fp=np.cumsum([0 if h else 1 for _,h in scored])
            recall=tp/n_refs
            precision=tp/np.maximum(tp+fp,1)
            precision=np.maximum.accumulate(precision[::-1])[::-1]
            sampled=np.zeros_like(recall_points)
            indices=np.searchsorted(recall,recall_points,side="left")
            valid=indices<len(precision)
            sampled[valid]=precision[indices[valid]]
            per_class[cls]=float(sampled.mean())
        per_threshold[threshold]=per_class

    means={t:(float(np.mean(list(v.values()))) if v else 0.0) for t,v in per_threshold.items()}
    return {
        "ap":float(np.mean(list(means.values()))) if means else 0.0,
        "ap50":means.get(0.5,0.0),
        "ap75":means.get(0.75,0.0),
        "per_phrase_ap50":per_threshold.get(0.5,{}),
    }

def operating_detection_metrics(predictions,records,threshold=0.10,iou=0.5):
    refs=[detection_reference(r) for r in records]
    tp=fp=fn=duplicates=0
    matched_ious=[]
    for dets,ref in zip(predictions,refs):
        claimed=[False]*len(ref["boxes"])
        filtered=[d for d in dets if float(d["score"])>=threshold]
        for det in sorted(filtered,key=lambda d:-float(d["score"])):
            best,best_iou=-1,0.0
            same_candidates=0
            for j,(box,label) in enumerate(zip(ref["boxes"],ref["labels"])):
                if label!=det["label"]:continue
                value=box_iou(det["box"],box)
                if value>=iou:same_candidates+=1
                if not claimed[j] and value>best_iou:
                    best,best_iou=j,value
            if best>=0 and best_iou>=iou:
                claimed[best]=True;tp+=1;matched_ious.append(best_iou)
            else:
                fp+=1
                if same_candidates:duplicates+=1
        fn+=sum(not x for x in claimed)
    return {
        "precision50":tp/(tp+fp) if tp+fp else 0.0,
        "recall50":tp/(tp+fn) if tp+fn else 0.0,
        "mean_matched_iou":float(np.mean(matched_ious)) if matched_ious else 0.0,
        "false_positives_per_image":fp/len(records),
        "missed_objects_per_image":fn/len(records),
        "duplicate_detections_per_image":duplicates/len(records),
    }

def common_classwise_nms(detections,iou=0.5):
    kept=[]
    for label in sorted({d["label"] for d in detections}):
        group=[d for d in detections if d["label"]==label]
        if not group:continue
        boxes=torch.tensor([d["box"] for d in group],dtype=torch.float32)
        scores=torch.tensor([d["score"] for d in group],dtype=torch.float32)
        indices=nms(boxes,scores,float(iou)).tolist()
        kept.extend(group[i] for i in indices)
    return sorted(kept,key=lambda d:-float(d["score"]))

empty=[[] for _ in validation_records]
assert common_ap(empty,validation_records)["ap"]==0

# 6. Counting metrics

Count error alone can hide compensating mistakes.

The common counting evaluator therefore reports:

- MAE
- RMSE
- NAE
- exact-count fraction
- under-count fraction
- over-count fraction
- point precision/recall/F1
- box precision/recall/F1 @ IoU 0.5
- mean matched IoU

In [ ]:
# @title Counting and localisation metrics
from scipy.optimize import linear_sum_assignment

def count_metrics(rows):
    errors=np.asarray([r["predicted"]-r["gold"] for r in rows],dtype=float)
    gold=np.asarray([r["gold"] for r in rows],dtype=float)
    return {
        "mae":float(np.mean(np.abs(errors))),
        "rmse":float(np.sqrt(np.mean(errors**2))),
        "nae":float(np.mean(np.abs(errors)/np.maximum(gold,1))),
        "exact_fraction":float(np.mean(errors==0)),
        "under_count_fraction":float(np.mean(errors<0)),
        "over_count_fraction":float(np.mean(errors>0)),
    }

def match_boxes(predicted,gold,iou_threshold=0.5):
    if not predicted and not gold:
        return {"tp":0,"fp":0,"fn":0,"ious":[]}
    if not predicted:
        return {"tp":0,"fp":0,"fn":len(gold),"ious":[]}
    if not gold:
        return {"tp":0,"fp":len(predicted),"fn":0,"ious":[]}
    matrix=np.asarray([[box_iou(p,g) for g in gold] for p in predicted])
    cost=1-matrix
    rows,cols=linear_sum_assignment(cost)
    matched=[matrix[r,c] for r,c in zip(rows,cols) if matrix[r,c]>=iou_threshold]
    tp=len(matched)
    return {"tp":tp,"fp":len(predicted)-tp,"fn":len(gold)-tp,"ious":matched}

def point_radius(record):
    sides=[
        ((b[2]-b[0])+(b[3]-b[1]))/2
        for b in record["target_boxes"][:3]
    ]
    return max(4.0,float(np.mean(sides))/2 if sides else 4.0)

def match_points(predicted,gold,radius):
    if not predicted and not gold:return {"tp":0,"fp":0,"fn":0}
    if not predicted:return {"tp":0,"fp":0,"fn":len(gold)}
    if not gold:return {"tp":0,"fp":len(predicted),"fn":0}
    p=np.asarray(predicted,float);g=np.asarray(gold,float)
    d=np.sqrt(((p[:,None,:]-g[None,:,:])**2).sum(axis=2))
    rows,cols=linear_sum_assignment(d)
    tp=sum(float(d[r,c])<=radius for r,c in zip(rows,cols))
    return {"tp":int(tp),"fp":len(predicted)-int(tp),"fn":len(gold)-int(tp)}

def prf(rows):
    tp=sum(r["tp"] for r in rows);fp=sum(r["fp"] for r in rows);fn=sum(r["fn"] for r in rows)
    p=tp/(tp+fp) if tp+fp else 0.0
    r=tp/(tp+fn) if tp+fn else 0.0
    f=2*p*r/(p+r) if p+r else 0.0
    return {"precision":p,"recall":r,"f1":f}

def evaluate_count_predictions(per_scene,records):
    count_rows=[]
    box_rows=[]
    point_rows=[]
    all_ious=[]
    for pred,record in zip(per_scene,records):
        count_rows.append({
            "id":record["id"],"gold":record["target_count"],"predicted":int(pred["count"])
        })
        boxes=match_boxes(pred.get("boxes",[]),record["target_boxes"])
        points=match_points(pred.get("points",[]),record["target_points"],point_radius(record))
        box_rows.append(boxes);point_rows.append(points);all_ious.extend(boxes["ious"])
    return {
        **count_metrics(count_rows),
        "boxes":{**prf(box_rows),"mean_matched_iou":float(np.mean(all_ious)) if all_ious else 0.0},
        "localisation":prf(point_rows),
        "per_scene":count_rows,
    }

# 7. Stage exact frozen detection snapshots

Grounding DINO and OWLv2 are normal Transformers-format repositories, so the frozen comparison does not depend on DIMER repository code.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
# @title Stage and verify HF snapshots
from huggingface_hub import hf_hub_download

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""):
            h.update(chunk)
    return h.hexdigest()

def stage_hf_manifest(manifest,root):
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    for name,size,digest in manifest["files"]:
        path=Path(hf_hub_download(
            repo_id=manifest["modelId"],
            filename=name,
            revision=manifest["revision"],
            local_dir=str(root),
        ))
        if path.stat().st_size!=size or sha256_file(path)!=digest:
            raise ValueError(f"{manifest['modelId']} {name}: integrity mismatch")
    # Write the carrier manifest too so FULL Grounding DINO can reuse this snapshot.
    normalized={
        "format":"dimer_hf_snapshot","formatVersion":1,
        "modelKey":manifest["modelKey"],"modelId":manifest["modelId"],
        "revision":manifest["revision"],
        "files":[{"path":n,"bytes":s,"sha256":d} for n,s,d in manifest["files"]],
        "totalBytes":sum(s for _,s,_ in manifest["files"]),
    }
    (root/"dimer-base-manifest.json").write_text(json.dumps(normalized,indent=2),encoding="utf-8")
    return root

GROUNDING_ROOT=stage_hf_manifest(GROUNDING_MANIFEST,WORK_ROOT/"models"/"grounding_dino")
OWLV2_ROOT=stage_hf_manifest(OWLV2_MANIFEST,WORK_ROOT/"models"/"owlv2")

print("Verified:",GROUNDING_ROOT,OWLV2_ROOT)

# 8. Grounding DINO and OWLv2 frozen wrappers

The primary detector comparison uses the exact bare labels from each scene:

```text
[target_label, distractor_label]
```

Grounding DINO reduces token-level logits into one score per supplied phrase and assigns every candidate its single best phrase.

OWLv2 already provides one image-text score per query phrase.

Both enter the common evaluator at the same score floor and candidate cap.

In [ ]:
# @title Frozen open-vocabulary detector wrappers
from transformers import (
    AutoModelForZeroShotObjectDetection,
    AutoProcessor,
    Owlv2ForObjectDetection,
    Owlv2Processor,
)

_SPECIAL_TOKEN_IDS=(101,102,1012,1029,0)

def grounding_text(prompts):
    cleaned=[p.strip().rstrip(".").strip().lower() for p in prompts]
    return " ".join(f"{p}." for p in cleaned),cleaned

def phrase_token_groups(input_ids):
    groups=[];current=[]
    for position,token in enumerate(input_ids):
        if int(token) in _SPECIAL_TOKEN_IDS:
            if current:
                groups.append(current);current=[]
        else:
            current.append(position)
    if current:groups.append(current)
    return groups

class GroundingFrozen:
    def __init__(self):
        self.processor=AutoProcessor.from_pretrained(
            str(GROUNDING_ROOT),local_files_only=True,trust_remote_code=False
        )
        self.model=AutoModelForZeroShotObjectDetection.from_pretrained(
            str(GROUNDING_ROOT),local_files_only=True,trust_remote_code=False
        ).to(DEVICE).eval()

    def predict(self,image,prompts,score_floor=DETECTION_SCORE_FLOOR):
        text,vocabulary=grounding_text(prompts)
        inputs=self.processor(images=image.convert("RGB"),text=text,return_tensors="pt")
        groups=phrase_token_groups(inputs["input_ids"][0].tolist())
        if len(groups)!=len(vocabulary):
            raise RuntimeError("Grounding DINO phrase/token grouping mismatch")
        inputs=inputs.to(DEVICE)
        with torch.inference_mode():
            out=self.model(**inputs)
        probs=out.logits[0].sigmoid().cpu()
        phrase_scores=torch.stack(
            [probs[:,group].max(dim=1).values for group in groups],dim=1
        )
        score,index=phrase_scores.max(dim=1)
        boxes=out.pred_boxes[0].cpu()
        w,h=image.size
        xyxy=torch.stack([
            (boxes[:,0]-boxes[:,2]/2)*w,
            (boxes[:,1]-boxes[:,3]/2)*h,
            (boxes[:,0]+boxes[:,2]/2)*w,
            (boxes[:,1]+boxes[:,3]/2)*h,
        ],dim=1)
        xyxy[:,0::2].clamp_(0,float(w))
        xyxy[:,1::2].clamp_(0,float(h))
        keep=score>=float(score_floor)
        dets=[
            {"box":[float(v) for v in b],"label":vocabulary[int(k)],"score":float(s)}
            for s,k,b in zip(score[keep],index[keep],xyxy[keep])
        ]
        return sorted(dets,key=lambda d:-d["score"])[:DETECTION_CANDIDATE_CAP]

class OWLv2Frozen:
    def __init__(self):
        self.processor=Owlv2Processor.from_pretrained(str(OWLV2_ROOT),local_files_only=True)
        self.model=Owlv2ForObjectDetection.from_pretrained(
            str(OWLV2_ROOT),local_files_only=True
        ).to(DEVICE).eval()

    def predict(self,image,prompts,score_floor=DETECTION_SCORE_FLOOR):
        queries=[p.strip().rstrip(".").strip().lower() for p in prompts]
        inputs=self.processor(images=image.convert("RGB"),text=[queries],return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            outputs=self.model(**inputs)
        result=self.processor.post_process_grounded_object_detection(
            outputs,
            threshold=float(score_floor),
            target_sizes=[image.size[::-1]],
            text_labels=[queries],
        )[0]
        dets=[
            {"box":[float(v) for v in box.tolist()],"label":str(label),"score":float(score)}
            for box,label,score in zip(
                result["boxes"],result["text_labels"],result["scores"]
            )
        ]
        return sorted(dets,key=lambda d:-d["score"])[:DETECTION_CANDIDATE_CAP]

# 9. Validation — frozen open-vocabulary detection

The primary raw-output comparison uses **no notebook-added NMS**.

A secondary diagnostic applies identical class-wise NMS to both systems to quantify how much duplicate boxes affect:

- detection metrics;
- detector-as-counter behavior.

In [ ]:
# @title Run Grounding DINO validation, then unload
import time
import gc

validation_detection={}
runtime_records={}

gd=GroundingFrozen()
t0=time.perf_counter()
validation_detection["grounding_dino"]=[
    gd.predict(r["image"],[r["target_label"],r["distractor_label"]])
    for r in validation_records
]
runtime_records["grounding_dino_validation_seconds"]=time.perf_counter()-t0
del gd
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

owl=OWLv2Frozen()
t0=time.perf_counter()
validation_detection["owlv2"]=[
    owl.predict(r["image"],[r["target_label"],r["distractor_label"]])
    for r in validation_records
]
runtime_records["owlv2_validation_seconds"]=time.perf_counter()-t0
del owl
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

rows=[]
for model,preds in validation_detection.items():
    raw={**common_ap(preds,validation_records),**operating_detection_metrics(preds,validation_records,0.10)}
    nms_preds=[common_classwise_nms(p,COMMON_NMS_IOU) for p in preds]
    post={**common_ap(nms_preds,validation_records),**operating_detection_metrics(nms_preds,validation_records,0.10)}
    rows.append({"model":model,"output":"raw",**raw})
    rows.append({"model":model,"output":"common_nms",**post})
validation_detection_table=pd.DataFrame(rows)
display(validation_detection_table)
validation_detection_table.to_csv(
    OUTPUT_ROOT/"detection"/"validation"/"metrics.csv",index=False
)

# 10. Validation prompt sensitivity and absent-prompt control

Prompt wording itself is an input variable.

Compare on validation:

- `{label}`
- `a photo of a {label}`

Also query an absent category:

`purple star`

The primary test remains on the bare phrases unless the experiment is explicitly re-specified before freeze.

In [ ]:
# @title Prompt sensitivity / absent prompt on a bounded validation subset
PROMPT_DIAGNOSTIC_RECORDS=validation_records[:4]

prompt_rows=[]
for model_name,Model in (("grounding_dino",GroundingFrozen),("owlv2",OWLv2Frozen)):
    model=Model()
    for template in ("bare","photo"):
        predictions=[]
        absent_counts=[]
        for r in PROMPT_DIAGNOSTIC_RECORDS:
            prompts=[
                r["target_label"] if template=="bare" else f"a photo of a {r['target_label']}",
                r["distractor_label"] if template=="bare" else f"a photo of a {r['distractor_label']}",
            ]
            pred=model.predict(r["image"],prompts)
            # Map templated labels back to the canonical labels for evaluation only.
            if template=="photo":
                mapping={prompts[0]:r["target_label"],prompts[1]:r["distractor_label"]}
                pred=[{**d,"label":mapping.get(d["label"],d["label"])} for d in pred]
            predictions.append(pred)
            absent=model.predict(r["image"],[ABSENT_PROMPT])
            absent_counts.append(sum(d["score"]>=0.10 for d in absent))
        values={**common_ap(predictions,PROMPT_DIAGNOSTIC_RECORDS),
                **operating_detection_metrics(predictions,PROMPT_DIAGNOSTIC_RECORDS,0.10)}
        prompt_rows.append({
            "model":model_name,"template":template,
            "absent_false_boxes_per_image":float(np.mean(absent_counts)),
            **values
        })
    del model
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

prompt_table=pd.DataFrame(prompt_rows)
display(prompt_table)
prompt_table.to_csv(
    OUTPUT_ROOT/"detection"/"validation"/"prompt_sensitivity.csv",index=False
)

# 11. Candidate-stage CountGD carrier bootstrap

The current release-grade CountGD carrier contains:

- pure-PyTorch CountGD/GroundingDINO model code;
- checkpoint pickle audit + one-time SafeTensors conversion;
- strict converted-weight loading;
- counting input validation;
- count/localization metrics;
- bounded adaptation;
- adapter artifact verification.

Until those modules are carried inline, this candidate notebook stages only the exact Python source files from the immutable DIMER carrier commit and verifies each Git blob identity before import.

No repository clone or editable install is performed.

In [ ]:
# @title Stage exact CountGD carrier source (candidate release gate)
import urllib.request

COUNTGD_CARRIER_COMMIT="8263e1e20ed3e2c9bf40dcca6a0ea998f4c14cab"
COUNTGD_SOURCE={
    "__init__.py":("d831590eeb7917806c50f5e0734a7486163bcc64",5038),
    "config.py":("3fab53a7040adb1158346afdb1b892c99f1a82f0",6334),
    "metrics.py":("335b8e4bb4a8f45da1329d3e84ff44cb6cd8302a",14757),
    "model.py":("3e5c099252d1b1566779cf9c5705db7f74592cb4",23154),
    "modeling.py":("51ff9d323751641fdb6f58ea3983e249203c39c3",158848),
    "pipeline.py":("ab839aadb21b6f7f6dd8af5ed9bd76bf631d2ce4",35418),
    "provenance.py":("7aa5daffaf2fb332706d0f0232fdbbaef1c3fe32",5847),
    "samples.py":("5a492d33c70360b0464a4f7d7569cbe71bb3d52b",96655),
    "synthetic.py":("20694c1708838847be34c0010acb51d1037815a8",6004),
}

def git_blob_sha1(payload):
    return hashlib.sha1(
        f"blob {len(payload)}\0".encode()+payload,
        usedforsecurity=False,
    ).hexdigest()

pkg_root=WORK_ROOT/"carrier_src"
countgd_pkg=pkg_root/"countgd_pipeline"
countgd_pkg.mkdir(parents=True,exist_ok=True)

for name,(blob_sha,size) in COUNTGD_SOURCE.items():
    url=(
        "https://raw.githubusercontent.com/kurtvalcorza/"
        f"countgd-object-counting-pipeline/{COUNTGD_CARRIER_COMMIT}/"
        f"src/countgd_pipeline/{name}"
    )
    req=urllib.request.Request(url,headers={"User-Agent":"DIMER-workshop/1.0"})
    with urllib.request.urlopen(req,timeout=90) as response:
        payload=response.read()
    if len(payload)!=size or git_blob_sha1(payload)!=blob_sha:
        raise ValueError(f"CountGD carrier source integrity mismatch: {name}")
    (countgd_pkg/name).write_bytes(payload)

if str(pkg_root.resolve()) not in sys.path:
    sys.path.insert(0,str(pkg_root.resolve()))

print({
    "carrier_commit":COUNTGD_CARRIER_COMMIT,
    "files":len(COUNTGD_SOURCE),
    "release_gate":"inline these verified modules before promotion",
})

## Stage CountGD model/tokenizer manifests

The carrier then uses its own qualified asset path:

- stage pinned source checkpoint;
- verify bytes/SHA-256;
- statically audit the pickle;
- convert once;
- serve/load SafeTensors only thereafter.

In [ ]:
# @title Prepare CountGD model/tokenizer directories
COUNTGD_WEIGHTS=WORK_ROOT/"models"/"countgd"
COUNTGD_BERT=WORK_ROOT/"models"/"bert"
COUNTGD_WEIGHTS.mkdir(parents=True,exist_ok=True)
COUNTGD_BERT.mkdir(parents=True,exist_ok=True)

(COUNTGD_WEIGHTS/"dimer-base-manifest.json").write_text(
    json.dumps(COUNTGD_BASE_MANIFEST,indent=2),encoding="utf-8"
)
(COUNTGD_BERT/"dimer-base-manifest.json").write_text(
    json.dumps(COUNTGD_BERT_MANIFEST,indent=2),encoding="utf-8"
)

from countgd_pipeline.pipeline import CountGDPipeline
from countgd_pipeline.metrics import template_matching_baseline

countgd=CountGDPipeline.from_pretrained(
    device=DEVICE,
    weights_dir=COUNTGD_WEIGHTS,
    tokenizer_dir=COUNTGD_BERT,
    allow_download=True,
)

print({
    "device":str(countgd.device),
    "weight_sha256":countgd.weight_sha256,
})

# 12. Validation — CountGD three prompt modes

The same validation scenes are mapped into the CountGD record contract:

- `label` = target phrase
- `count` = target count
- `boxes` = target boxes
- `points` = target centers
- `exemplars` = three target boxes

Distractors remain in the image but are not target annotations.

In [ ]:
# @title Map shared records to CountGD records
def as_countgd_record(r):
    return {
        "id":r["id"],
        "image":r["image"],
        "label":r["target_label"],
        "count":r["target_count"],
        "boxes":r["target_boxes"],
        "points":r["target_points"],
        "exemplars":r["exemplars"],
    }

countgd_train=[as_countgd_record(r) for r in train_records]
countgd_validation=[as_countgd_record(r) for r in validation_records]
countgd_test=[as_countgd_record(r) for r in test_records]

countgd_validation_results={}
for name,use_text,use_exemplars in [
    ("text",True,False),
    ("exemplar",False,True),
    ("text+exemplar",True,True),
]:
    result=countgd.evaluate(
        countgd_validation,
        threshold=COUNTGD_THRESHOLD,
        use_text=use_text,
        use_exemplars=use_exemplars,
    )
    countgd_validation_results[name]=result

display(pd.DataFrame([
    {
        "mode":name,
        "mae":r["mae"],"rmse":r["rmse"],"nae":r["nae"],
        "exact":r["exact_fraction"],
        "point_f1":r["localisation"]["f1"],
        "box_f1":r["boxes"]["f1"],
    }
    for name,r in countgd_validation_results.items()
]))

# 13. Non-neural counting baselines

Two references:

1. **mean-count baseline** — one constant learned from training counts only;
2. **template matcher** — exemplar-based normalized cross-correlation from the current CountGD tutorial.

> **Before you run it:** predict whether this simple reference will be easy or difficult for the learned model(s) to beat. Record the baseline before interpreting the more complex result.

In [ ]:
# @title Count baselines
mean_train_count=round(float(np.mean([r["target_count"] for r in train_records])))

mean_rows=[
    {"id":r["id"],"gold":r["target_count"],"predicted":mean_train_count}
    for r in validation_records
]
mean_validation=count_metrics(mean_rows)

template_validation=template_matching_baseline(countgd_validation)

print({
    "mean_count":mean_train_count,
    "mean_validation":mean_validation,
    "template_validation":{
        k:template_validation[k] for k in ("mae","rmse","nae","exact_fraction")
    },
})

# 14. Validation — detectors used as counters

For each detector:

```text
target phrase only
→ candidate boxes
→ common class-wise NMS
→ score threshold
→ count boxes
```

The count threshold is selected **only on validation** by:

1. lowest MAE;
2. then lowest RMSE;
3. then the higher threshold.

In [ ]:
# @title Select detector counting thresholds on validation
def detector_count_predictions(predictions,records,threshold):
    per_scene=[]
    for dets,r in zip(predictions,records):
        target=[d for d in dets if d["label"]==r["target_label"] and d["score"]>=threshold]
        target=common_classwise_nms(target,COMMON_NMS_IOU)
        boxes=[d["box"] for d in target]
        points=[[(b[0]+b[2])/2,(b[1]+b[3])/2] for b in boxes]
        per_scene.append({"count":len(boxes),"boxes":boxes,"points":points})
    return per_scene

# The existing validation detection calls included both target+distractor.
counter_thresholds={}
counter_validation={}

for model,preds in validation_detection.items():
    rows=[]
    for threshold in COUNT_THRESHOLD_GRID:
        per_scene=detector_count_predictions(preds,validation_records,threshold)
        result=evaluate_count_predictions(per_scene,validation_records)
        rows.append({"threshold":threshold,**{
            k:result[k] for k in ("mae","rmse","nae","exact_fraction")
        }})
    table=pd.DataFrame(rows)
    best=sorted(
        rows,
        key=lambda x:(x["mae"],x["rmse"],-x["threshold"])
    )[0]
    counter_thresholds[model]=best["threshold"]
    counter_validation[model]=table
    print(model,"selected",best)
    display(table)

pd.concat(
    [table.assign(model=model) for model,table in counter_validation.items()],
    ignore_index=True,
).to_csv(OUTPUT_ROOT/"cross_capability"/"validation_detector_count_thresholds.csv",index=False)

# 15. FULL tier — bounded Grounding DINO + CountGD adaptation

`STANDARD` skips this entire section.

`FULL`:

### Grounding DINO

- vocabulary = all 12 color/shape phrases
- train decoder/reference-point/box/contrastive heads
- freeze visual backbone, BERT, feature enhancer, query selection
- 6 epochs
- AdamW, `5e-5`
- validation mAP50 selection

### CountGD

- train final two decoder layers + decoder norm + shared box head
- 4 epochs
- AdamW, `2e-4`
- validation MAE selection
- frozen epoch 0 remains eligible

OWLv2 remains frozen.

In [ ]:
# @title Optional FULL Grounding DINO carrier bootstrap + adaptation
grounding_artifact_dir=None
grounding_full_report=None

if WORKSHOP_TIER=="FULL":
    GROUNDING_CARRIER_COMMIT="216c43e474f3084631aa2a32fa833fdd619ad42b"
    GROUNDING_SOURCE={
        "__init__.py":("994c7e58a2a4fdf810b58836252111cd13482f24",3206),
        "metrics.py":("c17c77f1259144dbcb2f0888d85718d4d0bd89a2",7449),
        "pipeline.py":("28f0a33f74ed25a5a0dc8b6622f54cad51bd9ed2",36681),
        "samples.py":("394db2bd6ba0c657a107dcad8e12af130785b680",154327),
    }
    grounding_pkg=pkg_root/"grounding_dino_detection_pipeline"
    grounding_pkg.mkdir(parents=True,exist_ok=True)

    for name,(blob_sha,size) in GROUNDING_SOURCE.items():
        url=(
            "https://raw.githubusercontent.com/kurtvalcorza/"
            f"grounding-dino-detection-pipeline/{GROUNDING_CARRIER_COMMIT}/"
            f"src/grounding_dino_detection_pipeline/{name}"
        )
        req=urllib.request.Request(url,headers={"User-Agent":"DIMER-workshop/1.0"})
        with urllib.request.urlopen(req,timeout=90) as response:
            payload=response.read()
        if len(payload)!=size or git_blob_sha1(payload)!=blob_sha:
            raise ValueError(f"Grounding DINO carrier source integrity mismatch: {name}")
        (grounding_pkg/name).write_bytes(payload)

    from grounding_dino_detection_pipeline.pipeline import GroundingDINOPipeline

    vocabulary=sorted({
        f"{colour} {shape}"
        for colour in COLOURS
        for shape in SHAPES
    })

    def grounding_record(r):
        return {
            "id":r["id"],
            "image":r["image"],
            "boxes":r["target_boxes"]+r["distractor_boxes"],
            "labels":[r["target_label"]]*r["target_count"]+
                     [r["distractor_label"]]*r["distractor_count"],
        }

    gd_pipe=GroundingDINOPipeline.from_pretrained(
        device=DEVICE,
        weights_dir=GROUNDING_ROOT,
        allow_download=False,
    )
    grounding_full_report=gd_pipe.adapt(
        [grounding_record(r) for r in train_records],
        [grounding_record(r) for r in validation_records],
        vocabulary,
        epochs=GROUNDING_EPOCHS,
        lr=GROUNDING_LR,
        batch_size=GROUNDING_BATCH_SIZE,
        seed=0,
    )
    grounding_artifact_dir=OUTPUT_ROOT/"artifacts"/"grounding_dino"
    gd_pipe.save_artifact(
        grounding_artifact_dir,
        metadata={"workshop":"open-vocabulary-detection-and-counting"},
    )
    (OUTPUT_ROOT/"adaptation"/"grounding_dino"/"training.json").write_text(
        json.dumps(grounding_full_report,indent=2,default=str),encoding="utf-8"
    )
    del gd_pipe
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
else:
    print("STANDARD: Grounding DINO adaptation skipped.")

In [ ]:
# @title Optional FULL CountGD adaptation + adapter export
countgd_artifact_dir=None
countgd_full_report=None

if WORKSHOP_TIER=="FULL":
    countgd_full_report=countgd.adapt(
        countgd_train,
        countgd_validation,
        epochs=COUNTGD_EPOCHS,
        lr=COUNTGD_LR,
        trainable_layers=2,
        seed=0,
    )
    countgd_artifact_dir=OUTPUT_ROOT/"artifacts"/"countgd"
    countgd.save_artifact(
        countgd_artifact_dir,
        metadata={"workshop":"open-vocabulary-detection-and-counting"},
    )
    (OUTPUT_ROOT/"adaptation"/"countgd"/"training.json").write_text(
        json.dumps(countgd_full_report,indent=2,default=str),encoding="utf-8"
    )
    print({
        "best_epoch":countgd_full_report["best_epoch"],
        "n_trainable":countgd_full_report["n_trainable"],
    })
else:
    print("STANDARD: CountGD adaptation skipped.")

# 16. Freeze-before-test

The following choices are now fixed:

- shared dataset and test scene IDs;
- bare canonical prompt wording;
- detector score floor;
- candidate cap;
- common NMS rule;
- detector-as-counter thresholds selected on validation;
- CountGD threshold;
- adaptation configuration/artifacts when `FULL`.

No test observation may change them.

In [ ]:
# @title Freeze experiment
frozen={
    "notebook_spec":"2.1",
    "profile":"MULTI-CAPABILITY",
    "mode":"WORKSHOP",
    "tier":WORKSHOP_TIER,
    "dataset":{
        "sha256":dataset_manifest["dataset_sha256"],
        "split_sizes":SPLIT_SIZES,
        "test_ids":[r["id"] for r in test_records],
    },
    "grounding_dino":{
        "model_id":GROUNDING_MANIFEST["modelId"],
        "revision":GROUNDING_MANIFEST["revision"],
        "weight_sha256":"1a2412ef99bd74bcd3c2a246fa1e48581f8889a1300c9051974741314fc042f3",
        "prompt_template":"bare target/distractor phrase",
        "score_floor":DETECTION_SCORE_FLOOR,
    },
    "owlv2":{
        "model_id":OWLV2_MANIFEST["modelId"],
        "revision":OWLV2_MANIFEST["revision"],
        "weight_sha256":"e1e130b9e404cf91a75ad45644c1da9d7fa5284085eecc864266a6923efb99e7",
        "prompt_template":"bare target/distractor phrase",
        "score_floor":DETECTION_SCORE_FLOOR,
    },
    "countgd":{
        "base_revision":COUNTGD_BASE_MANIFEST["revision"],
        "converted_weight_sha256":"8e44867b951e3a4205d918e022b78bc5fea218fd17c1851b864a01c421d2d443",
        "threshold":COUNTGD_THRESHOLD,
        "prompt_modes":["text","exemplar","text+exemplar"],
    },
    "detection_evaluation":{
        "iou_thresholds":list(IOU_THRESHOLDS),
        "candidate_cap":DETECTION_CANDIDATE_CAP,
        "common_nms_iou":COMMON_NMS_IOU,
        "primary_output":"raw",
    },
    "detector_as_counter_thresholds":counter_thresholds,
    "full_adaptation":{
        "grounding_dino":grounding_full_report,
        "countgd":countgd_full_report,
    },
    "candidate_release_gate":"inline CountGD/Grounding carrier source modules before promotion",
}
freeze_path=OUTPUT_ROOT/"frozen"/"frozen_experiment.json"
freeze_path.write_text(json.dumps(frozen,indent=2,default=str),encoding="utf-8")
print("Frozen:",freeze_path)

# 17. Independent detection test

> **What to notice.** Read localization quality, missed targets, duplicate detections, and prompt sensitivity together. Do not treat raw confidence values from different detector families as directly comparable probabilities.

In [ ]:
# @title Frozen Grounding DINO test
test_detection={}

gd=GroundingFrozen()
t0=time.perf_counter()
test_detection["grounding_dino"]=[
    gd.predict(r["image"],[r["target_label"],r["distractor_label"]])
    for r in test_records
]
runtime_records["grounding_dino_test_seconds"]=time.perf_counter()-t0
del gd
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

owl=OWLv2Frozen()
t0=time.perf_counter()
test_detection["owlv2"]=[
    owl.predict(r["image"],[r["target_label"],r["distractor_label"]])
    for r in test_records
]
runtime_records["owlv2_test_seconds"]=time.perf_counter()-t0
del owl
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

test_detection_rows=[]
for model,preds in test_detection.items():
    raw={**common_ap(preds,test_records),**operating_detection_metrics(preds,test_records,0.10)}
    nms_preds=[common_classwise_nms(p,COMMON_NMS_IOU) for p in preds]
    post={**common_ap(nms_preds,test_records),**operating_detection_metrics(nms_preds,test_records,0.10)}
    test_detection_rows.append({"model":model,"output":"raw",**raw})
    test_detection_rows.append({"model":model,"output":"common_nms",**post})

test_detection_table=pd.DataFrame(test_detection_rows)
display(test_detection_table)
test_detection_table.to_csv(
    OUTPUT_ROOT/"detection"/"test"/"aggregate_metrics.csv",index=False
)

## Optional FULL adapted Grounding DINO test

This row remains separate from OWLv2 because OWLv2 has not been adapted.

In [ ]:
# @title FULL adapted Grounding DINO test
full_grounding_test=None

if WORKSHOP_TIER=="FULL":
    from grounding_dino_detection_pipeline.pipeline import GroundingDINOPipeline
    adapted_gd=GroundingDINOPipeline.from_pretrained(
        device=DEVICE,weights_dir=GROUNDING_ROOT,allow_download=False
    )
    adapted_gd.load_artifact(grounding_artifact_dir)

    vocabulary=sorted({
        f"{colour} {shape}"
        for colour in COLOURS
        for shape in SHAPES
    })
    adapted_predictions=[]
    for r in test_records:
        raw=adapted_gd.predict_boxes(
            r["image"],[r["target_label"],r["distractor_label"]],
            score_floor=DETECTION_SCORE_FLOOR
        )
        adapted_predictions.append([
            {"score":score,"label":label,"box":box}
            for score,label,box in raw
        ])
    full_grounding_test={
        **common_ap(adapted_predictions,test_records),
        **operating_detection_metrics(adapted_predictions,test_records,0.10),
    }
    print(full_grounding_test)
    del adapted_gd
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
else:
    print("STANDARD: adapted Grounding DINO test skipped.")

# 18. Independent counting test

> **What to notice.** Compare numerical count error with the underlying localization evidence. A correct final count can still come from the wrong detected instances, so inspect both the count metric and the qualitative scene output.

In [ ]:
# @title Frozen/adapted CountGD test
countgd_test_results={}

# If FULL, `countgd` is the selected adapted in-memory model.
# To preserve the frozen baseline, reconstruct a fresh frozen CountGD below.
if WORKSHOP_TIER=="FULL":
    frozen_countgd=CountGDPipeline.from_pretrained(
        device=DEVICE,
        weights_dir=COUNTGD_WEIGHTS,
        tokenizer_dir=COUNTGD_BERT,
        allow_download=False,
    )
else:
    frozen_countgd=countgd

for name,use_text,use_exemplars in [
    ("text",True,False),
    ("exemplar",False,True),
    ("text+exemplar",True,True),
]:
    countgd_test_results[f"frozen_{name}"]=frozen_countgd.evaluate(
        countgd_test,threshold=COUNTGD_THRESHOLD,
        use_text=use_text,use_exemplars=use_exemplars
    )

if WORKSHOP_TIER=="FULL":
    for name,use_text,use_exemplars in [
        ("text",True,False),
        ("exemplar",False,True),
        ("text+exemplar",True,True),
    ]:
        countgd_test_results[f"adapted_{name}"]=countgd.evaluate(
            countgd_test,threshold=COUNTGD_THRESHOLD,
            use_text=use_text,use_exemplars=use_exemplars
        )

display(pd.DataFrame([
    {
        "system":name,
        "mae":r["mae"],"rmse":r["rmse"],"nae":r["nae"],
        "exact":r["exact_fraction"],
        "point_f1":r["localisation"]["f1"],
        "box_f1":r["boxes"]["f1"],
    }
    for name,r in countgd_test_results.items()
]))

In [ ]:
# @title Baselines + detector-as-counter on independent test
mean_test=count_metrics([
    {"gold":r["target_count"],"predicted":mean_train_count}
    for r in test_records
])
template_test=template_matching_baseline(countgd_test)

detector_counter_test={}
for model,preds in test_detection.items():
    threshold=counter_thresholds[model]
    per_scene=detector_count_predictions(preds,test_records,threshold)
    detector_counter_test[model]=evaluate_count_predictions(per_scene,test_records)

comparison_rows=[
    {"system":"mean_count","prompt":"none",**{k:mean_test[k] for k in ("mae","rmse","nae","exact_fraction")}},
    {"system":"template_matcher","prompt":"exemplar",**{
        k:template_test[k] for k in ("mae","rmse","nae","exact_fraction")
    }},
]
for model,result in detector_counter_test.items():
    comparison_rows.append({
        "system":model+"_as_counter","prompt":"text",
        **{k:result[k] for k in ("mae","rmse","nae","exact_fraction")},
        "point_f1":result["localisation"]["f1"],
        "box_f1":result["boxes"]["f1"],
    })
for name,result in countgd_test_results.items():
    comparison_rows.append({
        "system":"countgd_"+name,
        "prompt":name.split("_",1)[-1],
        **{k:result[k] for k in ("mae","rmse","nae","exact_fraction")},
        "point_f1":result["localisation"]["f1"],
        "box_f1":result["boxes"]["f1"],
    })

cross_table=pd.DataFrame(comparison_rows)
display(cross_table)
cross_table.to_csv(
    OUTPUT_ROOT/"cross_capability"/"comparison.csv",index=False
)

# 19. Density and distractor diagnostics

A counter can fail for different reasons as scenes become denser.

Use training counts to define three target-density groups, then examine test MAE.

Also inspect error against the distractor/target ratio—especially for exemplar-only CountGD, where visual similarity can override semantic category.

In [ ]:
# @title Density groups and distractor ratios
train_counts=np.asarray([r["target_count"] for r in train_records])
q1,q2=np.quantile(train_counts,[1/3,2/3])

def density_group(count):
    return "low" if count<=q1 else "medium" if count<=q2 else "high"

diagnostic_rows=[]
for system,result in detector_counter_test.items():
    per_scene=detector_count_predictions(
        test_detection[system],test_records,counter_thresholds[system]
    )
    for pred,r in zip(per_scene,test_records):
        diagnostic_rows.append({
            "system":system+"_as_counter",
            "image_id":r["id"],
            "density":density_group(r["target_count"]),
            "target_count":r["target_count"],
            "distractor_ratio":r["distractor_count"]/r["target_count"],
            "abs_error":abs(pred["count"]-r["target_count"]),
        })

density_df=pd.DataFrame(diagnostic_rows)
if len(density_df):
    display(density_df.groupby(["system","density"])["abs_error"].mean().reset_index())
    density_df.to_csv(OUTPUT_ROOT/"counting"/"test"/"density_metrics.csv",index=False)

# 20. Deterministic qualitative galleries

In [ ]:
# @title Detection disagreement gallery
per_image=[]
for i,r in enumerate(test_records):
    gd_ap=common_ap([test_detection["grounding_dino"][i]],[r],iou_thresholds=(0.5,))["ap50"]
    owl_ap=common_ap([test_detection["owlv2"][i]],[r],iou_thresholds=(0.5,))["ap50"]
    per_image.append({"i":i,"id":r["id"],"gd_ap50":gd_ap,"owl_ap50":owl_ap,"difference":gd_ap-owl_ap})
per_image_df=pd.DataFrame(per_image)

chosen=[
    ("largest Grounding DINO advantage",int(per_image_df.loc[per_image_df["difference"].idxmax(),"i"])),
    ("largest OWLv2 advantage",int(per_image_df.loc[per_image_df["difference"].idxmin(),"i"])),
    ("highest density",int(np.argmax([r["target_count"] for r in test_records]))),
]

for title,idx in chosen:
    r=test_records[idx]
    fig,axes=plt.subplots(1,3,figsize=(15,5))
    for ax in axes:
        ax.imshow(r["image"]);ax.axis("off")
    axes[0].set_title("Ground truth")
    for box in r["target_boxes"]:
        x0,y0,x1,y1=box
        axes[0].add_patch(plt.Rectangle((x0,y0),x1-x0,y1-y0,fill=False,linewidth=1))
    for box in r["distractor_boxes"]:
        x0,y0,x1,y1=box
        axes[0].add_patch(plt.Rectangle((x0,y0),x1-x0,y1-y0,fill=False,linewidth=1,linestyle="--"))

    for ax,model in zip(axes[1:],("grounding_dino","owlv2")):
        ax.set_title(model)
        for d in test_detection[model][idx]:
            if d["score"]<0.10:continue
            x0,y0,x1,y1=d["box"]
            ax.add_patch(plt.Rectangle((x0,y0),x1-x0,y1-y0,fill=False,linewidth=1))
            ax.text(x0,y0,f"{d['label']} {d['score']:.2f}",fontsize=6)

    fig.suptitle(f"{title}: {r['id']}")
    plt.tight_layout()
    safe=title.lower().replace(" ","_")
    plt.savefig(OUTPUT_ROOT/"figures"/f"detection_{safe}.png",dpi=150,bbox_inches="tight")
    plt.show()

# 21. FULL tier FSC-147 regression check

Synthetic adaptation can improve synthetic shapes while harming ordinary photographic counting.

When `FULL`, fetch the CountGD carrier's digest-pinned FSC-147 sample and compare:

- frozen CountGD
- adapted CountGD

on its 24-image test split.

This is a **regression check**, not part of the Grounding DINO / OWLv2 detector comparison.

In [ ]:
# @title Optional FSC-147 regression check
fsc_regression=None

if WORKSHOP_TIER=="FULL":
    from countgd_pipeline.samples import fetch_sample_dataset

    fsc=fetch_sample_dataset(cache_dir=WORK_ROOT/"models"/"fsc147")
    fsc_test=fsc["test"]

    frozen_fsc=frozen_countgd.evaluate(fsc_test)
    adapted_fsc=countgd.evaluate(fsc_test)

    fsc_regression={
        "frozen":{
            k:frozen_fsc[k] for k in ("mae","rmse","nae","exact_fraction")
        },
        "adapted":{
            k:adapted_fsc[k] for k in ("mae","rmse","nae","exact_fraction")
        },
        "n":len(fsc_test),
    }
    print(fsc_regression)
else:
    print("STANDARD: FSC-147 regression check skipped.")

# 22. Fresh adapter reload verification (`FULL`)

A saved adapter is only useful if a fresh base can reconstruct the adapted behavior.

For Grounding DINO:

- verify adapter/base identity;
- reload into fresh base;
- compare one fixed scene's labels, boxes and scores.

For CountGD:

- reload fresh base + adapter;
- compare count, boxes, points and scores.

In [ ]:
# @title Optional adapter reload parity
reload_parity={}

if WORKSHOP_TIER=="FULL":
    # Grounding DINO
    fresh_gd=GroundingDINOPipeline.from_pretrained(
        device=DEVICE,weights_dir=GROUNDING_ROOT,allow_download=False
    )
    fresh_gd.load_artifact(grounding_artifact_dir)
    r=validation_records[0]
    a=fresh_gd.predict_boxes(
        r["image"],[r["target_label"],r["distractor_label"]],
        score_floor=DETECTION_SCORE_FLOOR
    )
    b=fresh_gd.predict_boxes(
        r["image"],[r["target_label"],r["distractor_label"]],
        score_floor=DETECTION_SCORE_FLOOR
    )
    reload_parity["grounding_dino"]={
        "same_length":len(a)==len(b),
        "max_score_diff":max((abs(x[0]-y[0]) for x,y in zip(a,b)),default=0.0),
    }

    # CountGD
    fresh_countgd=CountGDPipeline.from_artifact(
        countgd_artifact_dir,
        device=DEVICE,
        weights_dir=COUNTGD_WEIGHTS,
        tokenizer_dir=COUNTGD_BERT,
        allow_download=False,
    )
    rr=countgd_validation[0]
    before=countgd.count(
        rr["image"],text=rr["label"],exemplars=rr["exemplars"],threshold=COUNTGD_THRESHOLD
    )["results"][0]
    after=fresh_countgd.count(
        rr["image"],text=rr["label"],exemplars=rr["exemplars"],threshold=COUNTGD_THRESHOLD
    )["results"][0]
    reload_parity["countgd"]={
        "count_equal":before["count"]==after["count"],
        "boxes_equal":before["boxes"]==after["boxes"],
        "points_equal":before["points"]==after["points"],
        "scores_equal":before["scores"]==after["scores"],
    }
    print(reload_parity)
else:
    print("STANDARD: adapter reload verification not applicable.")

# 23. New-seed post-test sanity scenes

Generate scenes `4000`, `4001`, `4002`, outside every train/validation/test range.

These may be scored because generator truth is known, but they do not alter the primary test summary.

In [ ]:
# @title New scenes
new_records=[synthetic_scene(seed) for seed in (4000,4001,4002)]
for r in new_records:r["split"]="new"

# CountGD combined prompting.
new_countgd=[as_countgd_record(r) for r in new_records]
new_countgd_result=countgd.evaluate(
    new_countgd,threshold=COUNTGD_THRESHOLD,use_text=True,use_exemplars=True
)
print({
    "CountGD_text+exemplar_MAE":new_countgd_result["mae"],
    "new_scenes":[r["id"] for r in new_records],
})

# 24. BYOD contract

A measurable BYOD archive should contain:

```text
dataset.zip
├── scenes.csv
├── boxes.csv
└── images/
```

`scenes.csv`

```text
image_id,file,target_label,split
scene001,images/001.jpg,red apple,train
```

`boxes.csv`

```text
image_id,label,x0,y0,x1,y1,is_exemplar
scene001,red apple,10,20,60,80,true
scene001,red apple,80,20,130,80,false
scene001,green apple,200,50,250,110,false
```

Detection needs all labels/boxes.

Counting needs:

- one target label per scene;
- all target instances;
- optional exemplar marks.

Preferred split ownership is explicit `train` / `validation` / `test`.

### Privacy

Images, prompts, boxes and exemplar annotations remain inside the selected notebook runtime; they are not sent to DIMER workers or APIs. A hosted notebook is still an external compute environment.

Do not upload confidential, personal, biometric, surveillance, security-sensitive, restricted or proprietary imagery unless authorized.

# 25. Interpretation boundaries

### Open vocabulary is not unlimited understanding

A supplied phrase can still:

- be absent;
- be ambiguous;
- be truncated;
- match a look-alike region;
- produce duplicate boxes.

### Scores are not probabilities

A `0.8` from Grounding DINO is not probabilistically equivalent to `0.8` from OWLv2 or CountGD.

### Detector box count is not automatically an object count

Duplicates, low recall, NMS, thresholds and density all matter.

### A correct numerical count can still be wrong

A system can miss target objects and count distractors, producing the same final integer.

That is why this notebook reads:

- count error;
- point localization;
- box localization.

### Synthetic evidence

Flat rendered shapes are useful because ground truth is exact.

They do not establish performance on photographs, aerial imagery, microscopy, documents, people or industrial cameras.

### People and personal attributes

Open-vocabulary prompts can name people and personal attributes. The built-in exercise does not. Person-related uses require separate privacy, fairness, consent, threshold and application review.

### Query ceiling

CountGD and Grounding DINO use roughly 900 queries in one pass. Very dense scenes near that limit cannot be reliably counted without another strategy.

# 26. Try it yourselfs

### Exercise A — prompt wording

Why can `red circle` and `a photo of a red circle` produce different detections?

### Exercise B — negative prompts

What does a box returned for `purple star` prove?

Nothing by itself—the phrase being supplied is not evidence that the thing exists.

### Exercise C — detector as counter

When is “detect + NMS + box count” sufficient, and when might a dedicated counter help?

### Exercise D — exemplar-only prompting

Why can visual exemplars count visually similar distractors of another semantic category?

### Exercise E — compensating errors

How can MAE be zero while point/box F1 is poor?

### Exercise F — adaptation

If synthetic adaptation improves the shape scenes but worsens FSC-147, did adaptation “work”?

The answer depends on the deployment distribution and the regression policy.

# 27. Export provenance and report bundle

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
# @title Final provenance export
import datetime
import shutil

experiment_manifest={
    "notebook_spec":"2.1",
    "notebook_profile":"MULTI-CAPABILITY",
    "notebook_mode":"WORKSHOP",
    "workshop_revision":"0.1.0-candidate",
    "timestamp_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "tier":WORKSHOP_TIER,
    "dataset":{
        "sha256":dataset_manifest["dataset_sha256"],
        "split_sizes":SPLIT_SIZES,
        "seed_bases":SEED_BASE,
    },
    "models":{
        "grounding_dino":frozen["grounding_dino"],
        "owlv2":frozen["owlv2"],
        "countgd":frozen["countgd"],
    },
    "detection_test":test_detection_table.to_dict("records"),
    "counting_test":cross_table.to_dict("records"),
    "detector_as_counter_thresholds":counter_thresholds,
    "full_adaptation":{
        "grounding_dino":grounding_full_report,
        "countgd":countgd_full_report,
        "fsc147_regression":fsc_regression,
        "reload_parity":reload_parity,
    },
    "runtime":runtime_records,
    "candidate_release_gates":[
        "inline the pinned CountGD carrier modules currently source-staged from immutable Git blobs",
        "inline the pinned Grounding DINO adaptation carrier modules in FULL",
        "fresh T4 STANDARD Run all",
        "fresh T4 FULL Run all",
        "record peak VRAM and exact wall times",
    ],
    "evidence_scope":"44 deterministic synthetic target+distractor scenes with exact boxes, points, counts and exemplars",
}

(OUTPUT_ROOT/"provenance"/"experiment_manifest.json").write_text(
    json.dumps(experiment_manifest,indent=2,default=str),encoding="utf-8"
)
(OUTPUT_ROOT/"workshop_summary.json").write_text(
    json.dumps({
        "tier":WORKSHOP_TIER,
        "dataset_sha256":dataset_manifest["dataset_sha256"],
        "detection_test":test_detection_table.to_dict("records"),
        "counting_test":cross_table.to_dict("records"),
    },indent=2,default=str),encoding="utf-8"
)

bundle=shutil.make_archive(
    str(Path(OUTPUT_DIR).resolve())+"_DIMER_Open_Vocab_Detection_Counting_Report",
    "zip",
    root_dir=Path(OUTPUT_DIR).resolve(),
)
print({"bundle":bundle,"sha256":sha256_file(bundle)})

# 28. Troubleshooting

| Symptom | Likely cause | Corrective action |
|---|---|---|
| Grounding/OWLv2 digest mismatch | changed/incomplete snapshot | remove cache and retry; never bypass pins |
| CountGD carrier source blob mismatch | source did not match pinned commit | stop; never execute unverified source |
| CountGD conversion takes time/disk | one-time 1.25 GB source checkpoint → SafeTensors conversion | use canonical T4/storage envelope |
| OWLv2 returns duplicates | no native NMS | inspect raw results; use common NMS only in the diagnostic/count branch |
| absent phrase yields boxes | open-vocabulary false positive | treat scores as rankings; validate threshold locally |
| detector count is too high | duplicates / low threshold | inspect common-NMS and validation-selected count threshold |
| detector count is too low | recall failure / high threshold | inspect target recall and density |
| CountGD count correct but box F1 poor | compensating category/localization error | do not rely on count alone |
| exemplar-only over-counts distractors | visual similarity lacks semantic constraint | add text or improve exemplar/domain validation |
| FULL synthetic adaptation hurts FSC-147 | domain-specific regression | retain frozen baseline or use explicit regression constraints |
| BYOD uses people/private imagery | governance issue, not a model error | process only when authorized and separately validated |

Explicit failure is preferable to silently changing prompts, thresholds, or test ownership.

## Try it yourself — one controlled change

Use the same experimental discipline as the canonical path:

**Predict → change one variable → rerun → observe → explain**

Use the existing prompt-sensitivity or density/distractor analysis. Predict how adding distractors or changing the prompt mode should affect detector box counts and CountGD's numerical count. Change one condition, rerun the relevant analysis, and explain whether a numerically correct count also corresponds to correct localization.

Keep exploratory changes separate from the frozen canonical test result.


## Self-paced checkpoint

Before opening the sample interpretation, answer:

1. What did the model/system receive as input, and what did it produce?
2. Which baseline/reference tells you whether the learned model added value?
3. What failure mode or tradeoff matters most here?
4. What additional evidence would you want before transferring the result to a new domain?

<details>
<summary><b>Show a sample interpretation</b></summary>

Counting and detection answer related but different questions. A detector's number of boxes is not automatically a reliable object count, and a numerically correct count can still be produced for the wrong spatial evidence. Inspect distractors, density, and prompt sensitivity together.

Use the outputs from **your run** when writing your final answer; small numeric differences across supported runtimes are possible.

</details>


## Write an evidence-based conclusion

1. **State the question** tested by this notebook.
2. **Report the primary result** against the relevant baseline/reference.
3. **Add supporting evidence** from a secondary metric, error pattern, disagreement, or qualitative diagnostic.
4. **Account for cost/complexity** when it materially affects the comparison.
5. **State the limits** of the data, split, model revision, and configuration.

Report detection and counting results separately, compare detector-as-counter behavior with the dedicated counting model and non-neural references, describe one density/distractor failure mode, and avoid treating one correct count as proof of correct visual grounding.


# Glossary

| Term | Meaning |
|---|---|
| **Open-vocabulary detection** | Object classes are supplied as text at inference rather than fixed by one classifier head |
| **Open-world counting** | Count instances of a prompted concept, including concepts not represented by one fixed class list |
| **Exemplar** | Box around a representative target instance used as a visual prompt |
| **AP50 / AP75** | Detection average precision at IoU ≥ 0.50 / 0.75 |
| **NMS** | Non-maximum suppression used here as a common de-duplication diagnostic |
| **MAE** | Mean absolute count error |
| **RMSE** | Root-mean-square count error |
| **NAE** | Count error normalized by the gold count |
| **Point F1** | One-to-one localization quality of predicted object centers |
| **Box F1** | One-to-one detection quality at a chosen IoU threshold |
| **Negative prompt** | Query for a concept known to be absent from the controlled sample |
| **Prompt sensitivity** | Model behavior changing when semantically similar wording changes |
| **Detector-as-counter** | Count after open-vocabulary detection, NMS and thresholding |
| **Sample-sanity evidence** | Bounded tutorial evidence, not deployment validation |

In [ ]:
# @title Run-all completion summary
summary={
    "notebook_spec":"2.1",
    "profile":"MULTI-CAPABILITY",
    "mode":"WORKSHOP",
    "tier":WORKSHOP_TIER,
    "models":["Grounding DINO Tiny","OWLv2 Base P16 Ensemble","CountGD"],
    "dataset_sha256":dataset_manifest["dataset_sha256"],
    "split_sizes":SPLIT_SIZES,
    "detector_count_thresholds":counter_thresholds,
    "output_directory":str(OUTPUT_ROOT.resolve()),
    "release_status":"candidate",
    "candidate_source_gate":"inline CountGD/Grounding carrier modules before promotion",
}
display(pd.Series(summary,name="value").to_frame())
print(
    "Open-vocabulary detection and counting workshop complete. "
    "Interpret all built-in results as synthetic sample-sanity evidence."
)

# Troubleshooting

| What you see | Likely cause | What to do |
|---|---|---|
| Accelerator unavailable or execution is unexpectedly slow | The runtime does not match the documented resource envelope | Select the documented accelerator/runtime, start a fresh session, and run top-to-bottom. |
| Package/version or stale-module error | Incompatible libraries were already imported in the hosted kernel | Start a fresh runtime and choose **Run all** before importing extra packages. Do not bypass version checks. |
| Model/sample digest or byte-size check fails | Download is incomplete or upstream bytes differ from the pinned artifact | Remove the affected runtime cache/download and rerun. Do not disable the integrity check. |
| Out-of-memory or runtime restart | Too many large models/intermediates are resident | Use the default tier, follow explicit unload/release steps, and avoid combining optional heavy branches. |
| BYOD validation fails | Input does not satisfy the documented schema, shape, labels, or limits | Follow the validation message, correct the indicated field/format, then rerun the BYOD branch. |
| Your numbers differ slightly | Supported hardware/library execution can introduce small numerical variation | Verify the split, model revision, metric definition, and qualitative pattern before treating the difference as substantive. |
